# View Creation: Assignment 2 
- Big Data - 2025
- Juan Sebastian Vargas Torres

## Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, LongType
import matplotlib.cbook as cbook
import matplotlib.pyplot as plt

## Utils

## Reading Data

In [0]:
catalog = "workspace"
source_schema  = "bde_silver"
taxis_table_name = 'taxis_trips'
TAXIS_TABLE_PATH = f"{catalog}.{source_schema}.{taxis_table_name}"
target_schema  = "bde_gold"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{target_schema}")

DataFrame[]

## View Creation

### 1a. Trips Counts
For each year and month: What was the total number of trips?

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{target_schema}.trips_per_month AS
SELECT
    DATE(DATE_TRUNC('month', pickup_datetime)) AS trip_month,
    COUNT(*) AS total_trips
FROM
    {TAXIS_TABLE_PATH}
GROUP BY
    trip_month
ORDER BY
    trip_month DESC
""")

DataFrame[]

In [0]:
spark.sql(f"""
SELECT *
FROM {catalog}.{target_schema}.trips_per_month
""").display()

trip_month,total_trips
2024-12-01,3084922
2024-11-01,3027957
2024-10-01,3175012
2024-09-01,2911619
2024-08-01,2480689
2024-07-01,2552495
2024-06-01,2897744
2024-05-01,3082795
2024-04-01,2861173
2024-03-01,2909090


### 1b. Top weekday
For each year and month: Which day of week (e.g. monday, tuesday, etc..) had the most trips?

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{target_schema}.top_weekdays AS
WITH month_weekday_count AS (
  SELECT
      DATE(DATE_TRUNC('month', pickup_datetime)) AS trip_month, 
      DATE_FORMAT(pickup_datetime, 'EEEE') AS weekday,
      COUNT(*) AS total_trips
  FROM
      {TAXIS_TABLE_PATH}
  GROUP BY
      trip_month, weekday
  ORDER BY
      trip_month DESC
),
top_weekdays AS (
  SELECT
      trip_month,
      weekday,
      total_trips,
      ROW_NUMBER() over (PARTITION BY trip_month ORDER BY total_trips DESC) as rank
  FROM
      month_weekday_count 
)
SELECT 
    trip_month,
    weekday,
    total_trips
FROM 
    top_weekdays
WHERE 
    rank = 1
ORDER BY 
    trip_month DESC
""")

DataFrame[]

In [0]:
spark.sql(f"""
SELECT *
FROM {catalog}.{target_schema}.top_weekdays
""").display()

trip_month,weekday,total_trips
2024-12-01,Tuesday,483085
2024-11-01,Saturday,539915
2024-10-01,Thursday,551843
2024-09-01,Thursday,447006
2024-08-01,Thursday,449305
2024-07-01,Wednesday,465827
2024-06-01,Saturday,479494
2024-05-01,Thursday,558606
2024-04-01,Tuesday,481665
2024-03-01,Friday,499487


### 1c. Top hours
For each year and month: Which hour of the day had the most trips?

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{target_schema}.top_hours AS
WITH month_hour_count AS (
  SELECT
      DATE(DATE_TRUNC('month', pickup_datetime)) AS trip_month, 
      HOUR(pickup_datetime) AS trip_hour,
      COUNT(*) AS total_trips
  FROM
      {TAXIS_TABLE_PATH}
  GROUP BY
      trip_month, trip_hour
  ORDER BY
      trip_month DESC
),
top_hours AS (
  SELECT
      trip_month,
      trip_hour,
      total_trips,
      ROW_NUMBER() over (PARTITION BY trip_month ORDER BY total_trips DESC) as rank
  FROM
      month_hour_count 
)
SELECT 
    trip_month,
    trip_hour,
    total_trips
FROM 
    top_hours
WHERE 
    rank = 1
ORDER BY 
    trip_month DESC
""")

DataFrame[]

In [0]:
spark.sql(f"""
SELECT *
FROM {catalog}.{target_schema}.top_hours
""").display()

trip_month,trip_hour,total_trips
2024-12-01,18,224808
2024-11-01,18,225126
2024-10-01,18,238530
2024-09-01,18,216068
2024-08-01,18,182604
2024-07-01,18,187231
2024-06-01,18,211468
2024-05-01,18,227176
2024-04-01,18,209395
2024-03-01,18,208686


### 1d. Average Passangers
For each year and month: What was the average number of passengers?

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{target_schema}.avg_passagers_per_month AS
  SELECT
      DATE(DATE_TRUNC('month', pickup_datetime)) AS trip_month,
      ROUND(AVG(passenger_count), 2) AS avg_pass
  FROM
      {TAXIS_TABLE_PATH}
  GROUP BY
      trip_month
  ORDER BY
      trip_month DESC
""")

DataFrame[]

In [0]:
spark.sql(f"""
SELECT *
FROM {catalog}.{target_schema}.avg_passagers_per_month
""").display()

trip_month,avg_pass
2024-12-01,1.37
2024-11-01,1.34
2024-10-01,1.32
2024-09-01,1.31
2024-08-01,1.36
2024-07-01,1.36
2024-06-01,1.35
2024-05-01,1.34
2024-04-01,1.35
2024-03-01,1.35


### 1e. Average amount paid
For each year and month: What was the average amount paid per trip (using total_amount)?

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{target_schema}.avg_amount_per_month AS
  SELECT
      DATE(DATE_TRUNC('month', pickup_datetime)) AS trip_month,
      ROUND(AVG(total_amount), 2) AS avg_amount
  FROM
      {TAXIS_TABLE_PATH}
  GROUP BY
      trip_month
  ORDER BY
      trip_month DESC
""")

DataFrame[]

In [0]:
spark.sql(f"""
SELECT *
FROM {catalog}.{target_schema}.avg_amount_per_month
""").display()

trip_month,avg_amount
2024-12-01,27.29
2024-11-01,26.6
2024-10-01,27.43
2024-09-01,27.8
2024-08-01,26.7
2024-07-01,26.44
2024-06-01,26.83
2024-05-01,27.13
2024-04-01,26.08
2024-03-01,25.7


### 1f. Average amount paid per passanger
For each year and month: What was the average amount paid per passenger (using total_amount)?

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{target_schema}.avg_amount_per_pass AS
  SELECT
      DATE(DATE_TRUNC('month', pickup_datetime)) AS trip_month,
      ROUND(AVG(total_amount/passenger_count), 2) AS avg_amount_per_pass
  FROM
      {TAXIS_TABLE_PATH}
  GROUP BY
      trip_month
  ORDER BY
      trip_month DESC
""")

DataFrame[]

In [0]:
spark.sql(f"""
SELECT *
FROM {catalog}.{target_schema}.avg_amount_per_pass
""").display()

trip_month,avg_amount_per_pass
2024-12-01,23.33
2024-11-01,23.1
2024-10-01,23.93
2024-09-01,24.36
2024-08-01,22.84
2024-07-01,22.72
2024-06-01,23.22
2024-05-01,23.59
2024-04-01,22.62
2024-03-01,22.3


### 2. Taxi Colour Stats
In a Single table/dataframe/output
For each taxi colour (yellow and green):
* a. What was the average, median, minimum and maximum trip duration in minutes (with 2 decimals, eg. 90 seconds = 1.50 min)?
* b. What was the average, median, minimum and maximum trip distance in km?
* c. What was the average, median, minimum and maximum speed in km per hour?

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{target_schema}.taxis_colour_distance_duration_speed_stats AS
    SELECT
        taxi_type,
    
        ROUND(avg(duration_in_minutes), 2) AS avg_duration_min,
        ROUND(percentile_approx(duration_in_minutes, 0.5), 2) AS median_duration_min,
        ROUND(min(duration_in_minutes), 2) AS min_duration_min,
        ROUND(max(duration_in_minutes), 2) AS max_duration_min,
    
        ROUND(avg(trip_distance) * 1.60934, 2) AS avg_distance_km,
        ROUND(percentile_approx(trip_distance, 0.5) * 1.60934, 2) AS median_distance_km,
        ROUND(min(trip_distance) * 1.60934, 2) AS min_distance_km,
        ROUND(max(trip_distance) * 1.60934, 2) AS max_distance_km,

        ROUND((SUM(trip_distance) / SUM(duration_in_hours)) * 1.60934, 2) AS avg_speed_kmh,
        ROUND(percentile_approx(speed, 0.5) * 1.60934, 2) AS median_speed_kmh,
        ROUND(min(speed) * 1.60934, 2) AS min_speed_kmh,
        ROUND(max(speed) * 1.60934, 2) AS max_speed_kmh
        
    FROM
        {TAXIS_TABLE_PATH}
    WHERE
        duration_in_hours > 0
    GROUP BY
      taxi_type
""")

DataFrame[]

In [0]:
spark.sql(f"""
SELECT *
FROM {catalog}.{target_schema}.taxis_colour_distance_duration_speed_stats
""").display()

taxi_type,avg_duration_min,median_duration_min,min_duration_min,max_duration_min,avg_distance_km,median_distance_km,min_distance_km,max_distance_km,avg_speed_kmh,median_speed_kmh,min_speed_kmh,max_speed_kmh
green,13.44,10.42,1.0,180.0,4.55,3.06,0.16,64.37,20.32,18.39,0.06,48.28
yellow,13.93,10.95,1.0,180.0,4.19,2.66,0.16,64.37,18.02,16.09,0.05,40.23


### 3. Taxi Colour pickup, dropoff stats
In a Single table/dataframe/output
For each taxi colour (yellow and green), each pair of pick up and drop off locations (use boroughs not the id), each month, each day of week and each hours:
* a. What was the total number of trips?
* b. What was the average distance?
* c. What was the average amount paid per trip (using total_amount)?
* d. What was the total amount paid (using total_amount)?

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{target_schema}.taxis_colour_pickup_dropoff_stats AS
    SELECT
        taxi_type,
        pickup_borough,
        dropoff_borough,
        month(pickup_datetime),
        DATE_FORMAT(pickup_datetime, 'EEEE') AS weekday,
        hour(pickup_datetime),

        COUNT(*) AS total_trips,
        ROUND(avg(trip_distance), 2) AS avg_distance,
        ROUND(avg(total_amount), 2) AS avg_amount,
        ROUND(sum(total_amount), 2) AS sum_total_amount
        
    FROM
        {TAXIS_TABLE_PATH}
    GROUP BY
      1,2,3,4,5,6
""")

DataFrame[]

In [0]:
spark.sql(f"""
SELECT *
FROM {catalog}.{target_schema}.taxis_colour_pickup_dropoff_stats    
""").display()

taxi_type,pickup_borough,dropoff_borough,month(pickup_datetime),weekday,hour(pickup_datetime),total_trips,avg_distance,avg_amount,sum_total_amount
yellow,Manhattan,EWR,11,Monday,7,216,17.36,95.98,20730.73
yellow,Brooklyn,Manhattan,11,Tuesday,6,2817,5.71,24.73,69659.97
yellow,Bronx,Manhattan,11,Wednesday,6,255,9.23,36.63,9341.71
yellow,Bronx,Bronx,11,Wednesday,7,281,2.71,16.54,4646.67
yellow,Bronx,Bronx,11,Wednesday,17,221,2.8,18.04,3987.59
yellow,Brooklyn,Manhattan,11,Friday,1,1098,4.9,21.92,24068.35
yellow,Brooklyn,Manhattan,12,Thursday,1,881,4.95,21.81,19214.39
yellow,Manhattan,Bronx,12,Saturday,16,1271,7.63,30.61,38905.2
yellow,Queens,Staten Island,10,Friday,17,46,27.07,102.26,4704.15
yellow,Manhattan,Manhattan,11,Wednesday,20,537367,1.95,14.03,7540888.61


### 4. Top 10 pickup→dropoff borough pairs revenue share
In a Single table/dataframe/output:
* For 2024, compute the share of total revenue contributed by the Top 10 pickup→dropoff borough pairs (ranked by total_amount).

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{target_schema}.top_pickup_dropoff_pairs_revenue_2024 AS
WITH
total_amount_2024 AS (
  SELECT 
    SUM(total_amount) AS overall_total_amount 
  FROM 
    {TAXIS_TABLE_PATH} 
  WHERE 
    YEAR(pickup_datetime) = 2024
),
top_10_trips AS (
  SELECT
    pickup_borough,
    dropoff_borough,
    SUM(total_amount) AS pair_total_amount        
  FROM
    {TAXIS_TABLE_PATH}
  WHERE 
    YEAR(pickup_datetime) = 2024
  GROUP BY
    1, 2
  ORDER BY 
    pair_total_amount DESC
  LIMIT 10
)
SELECT 
  top_10.pickup_borough,
  top_10.dropoff_borough,
  ROUND(top_10.pair_total_amount, 2) AS pair_total_amount,
  ROUND(top_10.pair_total_amount / total.overall_total_amount * 100, 2) AS revenue_share_percent
FROM 
  top_10_trips AS top_10,
  total_amount_2024 AS total
""")

DataFrame[]

In [0]:
spark.sql(f"""
SELECT *
FROM {catalog}.{target_schema}.top_pickup_dropoff_pairs_revenue_2024
""").display()

pickup_borough,dropoff_borough,pair_total_amount,revenue_share_percent
Manhattan,Manhattan,6.3219353083E8,69.9
Queens,Manhattan,1.1993762417E8,13.26
Manhattan,Queens,5.154058056E7,5.7
Manhattan,Brooklyn,3.057297796E7,3.38
Queens,Brooklyn,2.431170938E7,2.69
Queens,Queens,2.088889175E7,2.31
Brooklyn,Brooklyn,4876906.41,0.54
Manhattan,EWR,4621222.95,0.51
Brooklyn,Manhattan,4201664.21,0.46
Manhattan,Bronx,3996270.06,0.44


### 5. What was the percentage of trips where drivers received tips?

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{target_schema}.percentage_of_trips_with_tips AS
    SELECT
        (round(COUNT(CASE WHEN tip_amount > 0 THEN 1 END) * 100.0 / COUNT(*), 2)) AS percentage_with_tips
    FROM
        {TAXIS_TABLE_PATH}
""")

DataFrame[]

In [0]:
spark.sql(f"""
SELECT *
FROM {catalog}.{target_schema}.percentage_of_trips_with_tips
""").display()

percentage_with_tips
63.07


### 6. For trips where the driver received tips, what was the percentage where the driver received tips of at least $15?


In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{target_schema}.percentage_of_trips_with_at_least_15 AS
    SELECT
        round((COUNT(CASE WHEN tip_amount >= 15 THEN 1 END)/COUNT(CASE WHEN tip_amount > 0 THEN 1 END))*100,2) AS percentage_with_tips
    FROM
        {TAXIS_TABLE_PATH}
""")

DataFrame[]

In [0]:
spark.sql(f"""
SELECT *
FROM {catalog}.{target_schema}.percentage_of_trips_with_at_least_15
""").display()

percentage_with_tips
0.48


### 7. Classify each trip into bins of durations:
1. Under 5 Mins
2. From 5 mins to 10 mins
3. From 10 mins to 20 mins
4. From 20 mins to 30 mins
5. From 30 mins to 60 mins
6. At least 60 mins

Then for each bins, calculate: 
1. Average speed(km per hour)
2. Average distance per aud dollar(km per $)

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{target_schema}.duration_bins_stats AS          
    WITH binned_trips AS (
        SELECT
            CASE 
                WHEN duration_in_minutes < 5 THEN 'a. Under 5 Mins'
                WHEN duration_in_minutes >= 5 AND duration_in_minutes < 10 THEN 'b. 5 to 10 mins'
                WHEN duration_in_minutes >= 10 AND duration_in_minutes < 20 THEN 'c. 10 to 20 mins'
                WHEN duration_in_minutes >= 20 AND duration_in_minutes < 30 THEN 'd. 20 to 30 mins'
                WHEN duration_in_minutes >= 30 AND duration_in_minutes < 60 THEN 'e. 30 to 60 mins'
                ELSE 'f. At least 60 mins'
            END AS duration_bin,
            trip_distance * 1.60934 AS trip_distance_km,
            duration_in_hours,
            total_amount
        FROM
            {TAXIS_TABLE_PATH}
        WHERE
            duration_in_hours > 0 AND total_amount > 0
    )
    SELECT
        duration_bin,     
        ROUND(SUM(trip_distance_km) / SUM(duration_in_hours), 2) AS average_speed_kph,
        ROUND(AVG(trip_distance_km / total_amount), 2) AS average_km_per_dollar
    FROM
        binned_trips
    GROUP BY
        duration_bin
    ORDER BY
        duration_bin
""")

DataFrame[]

In [0]:
spark.sql(f"""
SELECT *
FROM {catalog}.{target_schema}.duration_bins_stats
""").display()

duration_bin,average_speed_kph,average_km_per_dollar
a. Under 5 Mins,18.87,0.16
b. 5 to 10 mins,16.83,0.21
c. 10 to 20 mins,16.64,0.25
d. 20 to 30 mins,18.02,0.29
e. 30 to 60 mins,21.8,0.34
f. At least 60 mins,21.41,0.49


### 8. Which duration bin will you advise a taxi driver to target to maximise his income?

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{target_schema}.duration_bins_profitability AS          
    WITH binned_trips AS (
    SELECT
        CASE 
            WHEN duration_in_minutes < 5 THEN 'a. Under 5 Mins'
            WHEN duration_in_minutes >= 5 AND duration_in_minutes < 10 THEN 'b. 5 to 10 mins'
            WHEN duration_in_minutes >= 10 AND duration_in_minutes < 20 THEN 'c. 10 to 20 mins'
            WHEN duration_in_minutes >= 20 AND duration_in_minutes < 30 THEN 'd. 20 to 30 mins'
            WHEN duration_in_minutes >= 30 AND duration_in_minutes < 60 THEN 'e. 30 to 60 mins'
            ELSE 'f. At least 60 mins'
        END AS duration_bin,
        total_amount,
        duration_in_hours,
        taxi_type
    FROM
        {TAXIS_TABLE_PATH}
    WHERE
        duration_in_hours > 0 AND total_amount > 0
)
SELECT
    taxi_type,
    duration_bin,
    ROUND(AVG(total_amount) / AVG(duration_in_hours), 2) AS income_per_hour,
    ROUND(AVG(total_amount), 2) AS average_fare
FROM
    binned_trips
GROUP BY
    taxi_type, duration_bin
ORDER BY
    taxi_type, income_per_hour DESC
""")

DataFrame[]

In [0]:
spark.sql(f"""
SELECT *
FROM {catalog}.{target_schema}.duration_bins_profitability
""").display()

taxi_type,duration_bin,income_per_hour,average_fare
green,a. Under 5 Mins,108.41,6.46
green,b. 5 to 10 mins,74.85,9.23
green,c. 10 to 20 mins,64.9,15.29
green,d. 20 to 30 mins,60.85,24.48
green,e. 30 to 60 mins,56.8,36.77
green,f. At least 60 mins,47.25,60.36
yellow,a. Under 5 Mins,124.06,7.35
yellow,b. 5 to 10 mins,83.06,10.28
yellow,c. 10 to 20 mins,66.41,15.64
yellow,e. 30 to 60 mins,65.88,43.54


**Suggestion for Green Taxi Drivers**

- To maximize their income, Green Taxi drivers should focus on finding as many short-duration fares as possible.
- Their most profitable trips are those under 5 minutes, earning them $108.48 per hour.
Their profitability drops with every subsequent duration bin, with long trips of 60+ minutes being the least profitable at only $47.73 per hour.

**Suggestion for Yellow Taxi Drivers**
-  Yellow Taxi drivers should follow the same strategy of targeting quick, short-distance trips.
The "Under 5 Mins" bin is their most lucrative category by a large margin, generating $124.13 per hour.
Similar to Green Taxis, their hourly income rate declines as trips get longer, with the lowest rate coming from trips that last over an hour.

It's also worth noting that Yellow Taxis consistently earn a higher income per hour than Green Taxis across all trip durations.